<a href="https://colab.research.google.com/github/scostavinicius/lean-agent/blob/main/lean_agent_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assistente de provas em Lean 4 — grafo LangGraph

**Dupla:** VINICIUS COSTA SOARES · MATHEUS VINÍCIUS SILVA FREIRE DE CASTRO

Tarefa da Unidade II — IA Agêntica (2026.2)

O código do agente vive no pacote `lean_agent/` do repositório; este notebook só o usa. O estado, os nós e as arestas do grafo estão descritos no README.

## 1 — Ambiente

In [ ]:
# Por quê: no Colab o repositório ainda não está na máquina. Fora dele, rode o
# notebook da raiz do repositório e esta célula só instala as dependências.
import os
if not os.path.exists("lean_agent"):
    !git clone -q https://github.com/scostavinicius/lean-agent.git
    %cd lean-agent
!pip install -q -r requirements.txt

In [ ]:
# Por quê: o Lean é o verificador do agente. O elan instala uma versão fixa do
# Lean, e lean_agent/lean.py acha o executável em ~/.elan/bin.
!curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.33.0
!~/.elan/bin/lean --version

In [ ]:
# Por quê: a tarefa proíbe a chave no repositório. Ela vem de um segredo do Colab
# chamado GROQ_API_KEY, de um .env (lido por lean_agent/modelo.py) ou é digitada aqui.
try:
    from google.colab import userdata
    os.environ.setdefault("GROQ_API_KEY", userdata.get("GROQ_API_KEY"))
except Exception as e:
    print(f"secret do Colab indisponível ({type(e).__name__}); tentando .env")

from dotenv import load_dotenv
load_dotenv()

if not os.environ.get("GROQ_API_KEY"):
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

## 2 — O grafo

In [ ]:
# Por quê: o desenho sai do grafo compilado, então é sempre o grafo que roda.
from IPython.display import Image, display

from lean_agent import criar_grafo, provar

grafo = criar_grafo()
try:
    display(Image(grafo.get_graph().draw_mermaid_png()))
except Exception as erro:  # o PNG usa o serviço mermaid.ink; sem rede, fica o texto
    print(f"sem PNG ({type(erro).__name__}); diagrama em Mermaid:")
    print(grafo.get_graph().draw_mermaid())

In [ ]:
# Por quê: o rastro mostra cada ferramenta que o agente escolheu e o que o Lean
# respondeu. É a evidência de que a decisão é dele, e de que quem encerra é o Lean.
from lean_agent.avaliacao import mostrar_rastro

estado = provar(grafo, "(a b : Nat) : a + b = b + a", "Teste1")
mostrar_rastro(estado)
print("
--- resposta ao usuário ---")
print(estado["messages"][-1].content)

In [ ]:
# Por quê: memória com checkpointer. Os dois turnos usam a mesma thread, e o
# segundo só faz sentido se o assistente lembrar do primeiro.
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "demo-memoria"}}
for texto in ["Prove que p ∧ q implica q ∧ p, para proposições p e q.",
              "Qual tática você usou na prova anterior?"]:
    estado = grafo.invoke({"messages": [HumanMessage(texto)]}, config)
    print(f"você> {texto}
assistente> {estado['messages'][-1].content}
")

## 3 — Casos de teste

In [ ]:
# Por quê: rodar os 10 casos pelo grafo e montar a tabela, com quanto cada um custou.
# Nos casos "falso", acertar é o Lean NÃO aceitar prova nenhuma.
from lean_agent.avaliacao import rodar_casos

resultados = rodar_casos(grafo=grafo)